# Block preconditioner vs SANEPIC's Jacobi preconditioner

pysanepic solves (PᵀN⁻¹P) m = PᵀN⁻¹d with preconditioned conjugate gradients
(PCG). Two preconditioners are available:

* `"jacobi"`: the original SANEPIC choice. The I part is the exact diagonal of
  PᵀN⁻¹P; Q and U use 2/diag_I, which assumes a uniform coverage of
  polarization angles in each pixel.
* `"block"` (default): the inverse of the exact 3×3 I/Q/U block of PᵀN⁻¹P in
  each pixel, built from the actual angles, polarization efficiency and 1/f
  noise correlation. For white noise it is exactly the inverse of the binned-map
  covariance.

Both converge to the same map; this notebook compares how fast, on one day of
LiteBIRD simulation without HWP (the case GLS map-making is designed for), for
two sets of detectors: a Q-type and a U-type T/B pair (good coverage of
polarization angles in each pixel) and a single T/B pair (poor coverage).

Q and U are solved only in pixels where they can be separated: at least 4 hits
and a polarization-angle coverage whose reciprocal condition number is at least
`min_pol_rcond` (default 1e-2); elsewhere only I is solved. The last section
shows what happens without this check.

Requirements: `pysanepic` and a `litebird_sim` version providing
`Simulation.make_sanepic_gls_map` (until [litebird_sim#568](https://github.com/litebird/litebird_sim/pull/568)
is merged: `pip install git+https://github.com/litebird/litebird_sim@sanepic_gls`).

In [ ]:
import time

import astropy.time
import healpy as hp
import litebird_sim as lbs
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
telescope, channel = "LFT", "L4-140"
# Without HWP the polarization angles come only from the detectors: compare a
# Q-type + U-type set (good angle coverage) with a single T/B pair (poor coverage)
configs = {
    "Q+U pairs": ["000_001_016_QA_140_T", "000_001_016_QA_140_B", "000_001_014_UA_140_T", "000_001_014_UA_140_B"],
    "single pair": ["000_001_016_QA_140_T", "000_001_016_QA_140_B"],
}
duration_days = 1.0
nside = 64
chunk_s = 3600.0
imo = lbs.Imo(flatfile_location=lbs.PTEP_IMO_LOCATION)


def simulate(detlist):
    sim = lbs.Simulation(
        base_path="preconditioner_output",
        imo=imo,
        start_time=astropy.time.Time("2030-01-01T00:00:00"),
        duration_s=duration_days * 86400,
        random_seed=12345,
    )
    sim.set_instrument(
        lbs.InstrumentInfo.from_imo(imo, f"/releases/vPTEP/satellite/{telescope}/instrument_info")
    )
    dets = [
        lbs.DetectorInfo.from_imo(
            url=f"/releases/vPTEP/satellite/{telescope}/{channel}/{d}/detector_info", imo=imo
        )
        for d in detlist
    ]
    sim.set_scanning_strategy(imo_url="/releases/vPTEP/satellite/scanning_parameters/")
    sim.create_observations(detectors=dets)
    sim.prepare_pointings()
    ch = lbs.FreqChannelInfo.from_imo(
        url=f"/releases/vPTEP/satellite/{telescope}/{channel}/channel_info", imo=imo
    )
    sim.get_sky(
        parameters=lbs.SkyGenerationParams(
            make_cmb=True, make_fg=False, seed_cmb=1, apply_beam=True,
            units="K_CMB", output_type="map", nside=nside,
        ),
        channels=[ch],
        store_in_observation=True,
    )
    sim.fill_tods()
    sim.add_noise(noise_type="one_over_f")
    return sim

## Run both preconditioners

The same stopping threshold is used for all runs.

In [ ]:
sims, results = {}, {}
for config, detlist in configs.items():
    sims[config] = simulate(detlist)
    for kind in ["jacobi", "block"]:
        t0 = time.time()
        res = sims[config].make_sanepic_gls_map(
            nside=nside, chunk_s=chunk_s, preconditioner=kind, append_to_report=False
        )
        label = f"{config}, {kind}"
        results[label] = (res, time.time() - t0)
        n_qu, n_i = np.sum(res.maps[1] != hp.UNSEEN), np.sum(res.maps[0] != hp.UNSEEN)
        print(f"{label:20s} {res.iterations:4d} iterations  {results[label][1]:6.1f} s  "
              f"converged: {res.converged}  Q/U solved in {n_qu}/{n_i} pixels")

In [ ]:
plt.figure(figsize=(8, 5))
for label, (res, _) in results.items():
    plt.semilogy(res.residuals, label=label, ls="-" if "block" in label else "--")
plt.xlabel("iteration")
plt.ylabel(r"$|r|^2 / |b|^2$")
plt.legend()
plt.title("PCG convergence");

## Same solution

The converged maps agree to a small fraction of the noise level; the I monopole
is removed because 1/f noise leaves it (almost) unconstrained.

In [ ]:
for config in configs:
    a, b = results[f"{config}, block"][0].maps, results[f"{config}, jacobi"][0].maps
    seen = (a[1] != hp.UNSEEN) & (b[1] != hp.UNSEEN)
    d = (a - b)[:, seen]
    d[0] -= d[0].mean()
    print(f"{config:12s} max |block - jacobi| I/Q/U [uK]: " + " / ".join(f"{1e6 * np.abs(x).max():.2e}" for x in d))

## Pixels where Q and U cannot be separated

With a single T/B pair many pixels see too narrow a range of polarization
angles. With `min_pol_rcond=0` (the original SANEPIC criterion, only ≥ 4 hits)
Q and U are solved there too.

In [ ]:
sim = sims["single pair"]
t0 = time.time()
res0 = sim.make_sanepic_gls_map(nside=nside, chunk_s=chunk_s, min_pol_rcond=0.0, append_to_report=False)
res = results["single pair, block"][0]
extra = (res0.maps[1] != hp.UNSEEN) & (res.maps[1] == hp.UNSEEN)
ok = res.maps[1] != hp.UNSEEN
print(f"min_pol_rcond=0: {res0.iterations} iterations, {time.time() - t0:.1f} s; Q/U solved in {extra.sum()} more pixels")
print(f"rms Q in those pixels: {1e6 * np.std(res0.maps[1][extra]):.3g} uK, "
      f"in the others: {1e6 * np.std(res0.maps[1][ok]):.3g} uK")

## Summary

On this day of data:

| detectors | Jacobi | block | Q/U solved |
|---|---|---|---|
| Q+U pairs (4) | 56 iterations | 54 iterations | 19159 / 19159 pixels |
| single T/B pair | 101 iterations | 46 iterations | 6664 / 19159 pixels |

* With a good coverage of polarization angles (Q-type and U-type detectors),
  SANEPIC's Jacobi approximation already works and the two preconditioners are
  equivalent.
* With a poor coverage (a single T/B pair) the block preconditioner needs less
  than half the iterations.
* With a single pair, Q and U cannot be separated in about 2/3 of the pixels:
  with `min_pol_rcond=0` they are "solved" there anyway and the result is pure
  noise (rms Q ~ 0.1 K against ~40 µK elsewhere). The default `min_pol_rcond`
  solves only I in those pixels.